In [46]:
import pandas as pd
import numpy as np
import logging
import warnings
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

# Ignore deprecation warnings for cleaner terminal output
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class DataPreprocessor:
    def __init__(self, raw_data_path: str):
        self.raw_data_path = raw_data_path

    def process_data(self) -> pd.DataFrame:
        logger.info("Loading and preprocessing macroeconomic dataset...")
        df = pd.read_csv(self.raw_data_path, index_col='Date', parse_dates=True)

        df['Inflation_YoY'] = df['Inflation_CPI'].pct_change(12) * 100
        df['IndPro_YoY'] = df['Industrial_Production'].pct_change(12) * 100
        df = df.drop(columns=['Inflation_CPI', 'Industrial_Production'])

        # Apply 1-month lag to prevent data leakage
        macro_cols = ['Unemployment_Rate', 'Fed_Funds_Rate', 'Yield_Curve_Spread',
                      'VIX_Volatility', 'Inflation_YoY', 'IndPro_YoY']
        df[macro_cols] = df[macro_cols].shift(1)
        df = df.dropna()

        # Label regimes
        df['Target_Regime'] = df.apply(self._categorize_regime, axis=1)
        return df

    def _categorize_regime(self, row: pd.Series) -> int:
        if row['SPY_Return'] >= -0.04:
            return 0  # Normal
        elif row['SPY_Return'] < -0.04 and row['Inflation_YoY'] < 3.0:
            return 1  # Deflation
        else:
            return 2  # Inflation

class RegimeClassifier:
    def __init__(self, random_state: int = 42):
        self.rf_model = RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=random_state)
        self.xgb_model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=random_state)

    def train_evaluate_chronological(self, X: pd.DataFrame, y: pd.Series, test_size: float = 0.2):
        split_idx = int(len(X) * (1 - test_size))
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

        logger.info("Fitting Random Forest...")
        self.rf_model.fit(X_train, y_train)
        rf_preds = self.rf_model.predict(X_test)

        logger.info("Fitting XGBoost...")
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
        self.xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
        xgb_preds = self.xgb_model.predict(X_test)

        logger.info("\nRandom Forest Report:\n" + classification_report(y_test, rf_preds, zero_division=0))
        logger.info("\nXGBoost Report:\n" + classification_report(y_test, xgb_preds, zero_division=0))

        return pd.Series(rf_preds, index=X_test.index), pd.Series(xgb_preds, index=X_test.index)

class VectorizedBacktester:
    def __init__(self, asset_returns: pd.DataFrame, predictions: pd.DataFrame):
        self.data = asset_returns.join(predictions, how='inner')

    def run_simulation(self, pred_column: str, portfolio_name: str) -> pd.Series:
        conditions = [self.data[pred_column] == 0, self.data[pred_column] == 1, self.data[pred_column] == 2]
        allocations = [(0.60 * self.data['SPY_Return']) + (0.40 * self.data['TLT_Return']), self.data['TLT_Return'], self.data['GLD_Return']]
        self.data[portfolio_name] = np.select(conditions, allocations, default=0.0)
        return self.data[portfolio_name]

    def evaluate_metrics(self, returns: pd.Series, name: str):
        cum_wealth = (1 + returns).cumprod()
        tot_return = cum_wealth.iloc[-1] - 1
        max_drawdown = ((cum_wealth - cum_wealth.cummax()) / cum_wealth.cummax()).min()
        sortino = (returns.mean() * 12) / (returns[returns < 0].std() * np.sqrt(12))

        logger.info(f"{name} -> Return: {tot_return*100:.2f}% | Max DD: {max_drawdown*100:.2f}% | Sortino: {sortino:.2f}")


if __name__ == "__main__":

    # 1. Preprocessing & ML Evaluation
    preprocessor = DataPreprocessor('master_dataset.csv')
    df = preprocessor.process_data()

    X_empirical = df.drop(columns=['GLD_Return', 'SPY_Return', 'TLT_Return', 'Target_Regime'])
    y_empirical = df['Target_Regime']
    returns_empirical = df[['SPY_Return', 'TLT_Return', 'GLD_Return']]

    classifier = RegimeClassifier()
    rf_preds, xgb_preds = classifier.train_evaluate_chronological(X_empirical, y_empirical)
    predictions_df = pd.DataFrame({'RF_Prediction': rf_preds, 'XGB_Prediction': xgb_preds}, index=rf_preds.index)


    # 2. Financial Back-testing
    logger.info("\n--- PORTFOLIO PERFORMANCE ---")
    out_of_sample_returns = returns_empirical.loc[predictions_df.index]
    benchmark_returns = (0.60 * out_of_sample_returns['SPY_Return']) + (0.40 * out_of_sample_returns['TLT_Return'])

    backtester = VectorizedBacktester(out_of_sample_returns, predictions_df)
    backtester.run_simulation('RF_Prediction', 'RF_Portfolio')
    backtester.run_simulation('XGB_Prediction', 'XGB_Portfolio')

    backtester.evaluate_metrics(benchmark_returns, "60/40 Benchmark")
    backtester.evaluate_metrics(backtester.data['RF_Portfolio'], "Random Forest")
    backtester.evaluate_metrics(backtester.data['XGB_Portfolio'], "XGBoost")


    # 3. Generate Timeline
    latex_df = pd.DataFrame({
        'Actual_Regime': y_empirical.loc[predictions_df.index].astype(int),
        'RF_Predicted': predictions_df['RF_Prediction'].astype(int),
        'XGB_Predicted': predictions_df['XGB_Prediction'].astype(int)
    })
    latex_df.insert(0, 'X', range(1, len(latex_df) + 1))

    print("\n" + "="*50 + "\n[1] LATEX DATA: TIMELINE (PASTE INTO ALL 3 PANELS)\n" + "="*50)
    print(latex_df.to_string(index=False))

    # 4. Generate Drawdowns
    bench_wealth = (1 + benchmark_returns).cumprod()
    rf_wealth = (1 + backtester.data['RF_Portfolio']).cumprod()
    xgb_wealth = (1 + backtester.data['XGB_Portfolio']).cumprod()

    dd_df = pd.DataFrame({
        'Bench_DD': (((bench_wealth / bench_wealth.cummax()) - 1) * 100).values.round(2),
        'RF_DD': (((rf_wealth / rf_wealth.cummax()) - 1) * 100).values.round(2),
        'XGB_DD': (((xgb_wealth / xgb_wealth.cummax()) - 1) * 100).values.round(2)
    })
    dd_df.insert(0, 'X', range(1, len(dd_df) + 1))

    print("\n" + "="*50 + "\n[2] LATEX DATA: DRAWDOWNS (PASTE INTO ALL 3 PLOTS)\n" + "="*50)

    # Helper function to print data in dense horizontal chunks
    def print_dense_coords(df, y_column):
        coords = [f"({df['X'].iloc[i]}, {df[y_column].iloc[i]:.2f})" for i in range(len(df))]
        for i in range(0, len(coords), 10):  # Prints 10 coordinates per line
            print(" ".join(coords[i:i+10]))

    print("60/40 BENCHMARK")
    print_dense_coords(dd_df, 'Bench_DD')

    print("\nRANDOM FOREST")
    print_dense_coords(dd_df, 'RF_DD')

    print("\nXGBOOST")
    print_dense_coords(dd_df, 'XGB_DD')

    # 5. Generate Features
    def prep_features(model):
        df = pd.DataFrame({'Feature': X_empirical.columns, 'Importance': model.feature_importances_})
        df = df.sort_values('Importance', ascending=True)
        df.insert(0, 'Y', range(1, len(df) + 1))
        return df[['Y', 'Feature', 'Importance']]

    rf_features = prep_features(classifier.rf_model)
    xgb_features = prep_features(classifier.xgb_model)

    print("\n" + "="*50 + "\n[3] LATEX DATA: RF FEATURES\n" + "="*50)
    print("YTICKLABELS TO MANUALLY TYPE: " + ", ".join(rf_features['Feature'].str.replace('_', '\\_')))
    print("-" * 50)
    print(rf_features[['Y', 'Importance']].to_string(index=False))

    print("\n" + "="*50 + "\n[4] LATEX DATA: XGBOOST FEATURES\n" + "="*50)
    print("YTICKLABELS TO MANUALLY TYPE: " + ", ".join(xgb_features['Feature'].str.replace('_', '\\_')))
    print("-" * 50)
    print(xgb_features[['Y', 'Importance']].to_string(index=False))
    print("="*50 + "\n")

    # 6. Cumulative Wealth
    # Insert a starting value of 1.0 at X=0 so all lines originate from the exact same point
    bench_w = np.insert(bench_wealth.values, 0, 1.0)
    rf_w = np.insert(rf_wealth.values, 0, 1.0)
    xgb_w = np.insert(xgb_wealth.values, 0, 1.0)

    wealth_df = pd.DataFrame({
        'X': range(0, len(bench_w)),
        'Bench_Wealth': bench_w,
        'RF_Wealth': rf_w,
        'XGB_Wealth': xgb_w
    })

    print("\n" + "="*50 + "\n[5] LATEX DATA: CUMULATIVE WEALTH\n" + "="*50)

    # Helper function
    def print_dense_coords(df, y_column):
        coords = [f"({df['X'].iloc[i]}, {df[y_column].iloc[i]:.4f})" for i in range(len(df))]
        for i in range(0, len(coords), 10):
            print(" ".join(coords[i:i+10]))

    print("60/40 BENCHMARK")
    print_dense_coords(wealth_df, 'Bench_Wealth')

    print("\nRANDOM FOREST")
    print_dense_coords(wealth_df, 'RF_Wealth')

    print("\nXGBOOST")
    print_dense_coords(wealth_df, 'XGB_Wealth')

2026-09-01 21:56:07,730 - INFO - Loading and preprocessing macroeconomic dataset...
2026-09-01 21:56:07,743 - INFO - Fitting Random Forest...
2026-09-01 21:56:07,903 - INFO - Fitting XGBoost...
2026-09-01 21:56:08,464 - INFO - 
Random Forest Report:
              precision    recall  f1-score   support

           0       0.88      0.88      0.88        42
           1       0.00      0.00      0.00         2
           2       0.38      0.50      0.43         6

    accuracy                           0.80        50
   macro avg       0.42      0.46      0.44        50
weighted avg       0.79      0.80      0.79        50

2026-09-01 21:56:08,468 - INFO - 
XGBoost Report:
              precision    recall  f1-score   support

           0       0.85      0.83      0.84        42
           1       0.00      0.00      0.00         2
           2       0.22      0.33      0.27         6

    accuracy                           0.74        50
   macro avg       0.36      0.39      0.37    


[1] LATEX DATA: TIMELINE (PASTE INTO ALL 3 PANELS)
 X  Actual_Regime  RF_Predicted  XGB_Predicted
 1              2             2              0
 2              0             2              0
 3              2             2              2
 4              2             0              0
 5              0             2              0
 6              0             0              0
 7              2             2              2
 8              0             2              2
 9              0             2              2
10              0             0              2
11              0             0              2
12              0             0              2
13              0             0              2
14              0             0              0
15              0             0              0
16              2             0              0
17              0             0              2
18              0             0              0
19              0             0              0
20      